# Local Flask Runner

Use this notebook when running the project locally in VS Code/Jupyter, not in Google Colab.

Before using the app, make sure these trained checkpoint files exist:

- `checkpoints/covid_dcgan.pth`
- `checkpoints/covid_classifier.pth`

If they were trained in Colab, download/copy them from Drive into the local `checkpoints/` folder.


In [12]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Python version:', sys.version)

required_files = ['frontend/app.py', 'classifier.py', 'gan.py', 'data.py']
for path in required_files:
    print(path, 'OK' if Path(path).exists() else 'MISSING')


Project root: /home/mohssine/Desktop/ETUDES/DEEPLEARNING/medical_gan_project
Python executable: /home/mohssine/Desktop/ETUDES/DEEPLEARNING/medical_gan_project/.venv/bin/python
Python version: 3.11.15 (main, May 10 2026, 19:28:18) [Clang 22.1.3 ]
frontend/app.py OK
classifier.py OK
gan.py OK
data.py OK


In [13]:
from pathlib import Path
import shutil
import zipfile

project_checkpoints = Path('checkpoints')
project_checkpoints.mkdir(exist_ok=True)

search_roots = [
    Path.home() / 'Downloads',
    Path.home() / 'Desktop',
    Path.cwd(),
]

# First, look for direct .pth files.
for root in search_roots:
    if not root.exists():
        continue
    for name in ['covid_dcgan.pth', 'covid_classifier.pth']:
        matches = sorted(root.rglob(name), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            src = matches[0]
            dst = project_checkpoints / name
            if src.resolve() != dst.resolve():
                shutil.copy2(src, dst)
                print('Copied:', src, '->', dst)

# Then, look for the zip created by the Colab notebook.
if not all((project_checkpoints / name).exists() for name in ['covid_dcgan.pth', 'covid_classifier.pth']):
    zip_matches = []
    for root in search_roots:
        if root.exists():
            zip_matches.extend(root.rglob('medical_gan_checkpoints*.zip'))
            zip_matches.extend(root.rglob('*checkpoints*.zip'))
    zip_matches = sorted(set(zip_matches), key=lambda p: p.stat().st_mtime, reverse=True)
    if zip_matches:
        zip_path = zip_matches[0]
        print('Extracting:', zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(project_checkpoints)

for name in ['covid_dcgan.pth', 'covid_classifier.pth']:
    path = project_checkpoints / name
    print(path, 'OK' if path.exists() else 'MISSING')


checkpoints/covid_dcgan.pth OK
checkpoints/covid_classifier.pth OK


## Install Local Dependencies

Run this once for the selected notebook kernel. CPU-only PyTorch is used because it is lighter and enough for running Flask locally.


In [14]:
import importlib.util
import subprocess
import sys

packages = ['flask', 'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'kaggle', 'pillow']

if importlib.util.find_spec('pip') is None:
    print('pip is missing for this notebook kernel; trying ensurepip...')
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'torch',
    'torchvision',
    '--index-url',
    'https://download.pytorch.org/whl/cpu',
], check=True)

print('Dependencies are installed for:', sys.executable)



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


Dependencies are installed for: /home/mohssine/Desktop/ETUDES/DEEPLEARNING/medical_gan_project/.venv/bin/python



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


## Check Model Files

The app can open without checkpoints, but classification and generation need the `.pth` files below.


In [15]:
from pathlib import Path

Path('checkpoints').mkdir(exist_ok=True)

required_checkpoints = [
    Path('checkpoints/covid_dcgan.pth'),
    Path('checkpoints/covid_classifier.pth'),
]

missing = []
for path in required_checkpoints:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{path}: OK ({size_mb:.1f} MB)')
    else:
        print(f'{path}: MISSING')
        missing.append(path)

if missing:
    print('\nCopy these files into the checkpoints folder before using the model actions:')
    for path in missing:
        print(' -', path)
else:
    print('\nAll checkpoints are ready.')


checkpoints/covid_dcgan.pth: OK (25.8 MB)
checkpoints/covid_classifier.pth: OK (33.5 MB)

All checkpoints are ready.


## Start Flask Locally

Run this cell, then open the printed local links in your browser. The server keeps running in the background.


In [16]:
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PREFERRED_PORT = 5000
BIND_HOST = '0.0.0.0'
VISIT_HOST = '127.0.0.1'


def find_free_port(start_port=5000, attempts=20):
    for port in range(start_port, start_port + attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(0.2)
            if sock.connect_ex((VISIT_HOST, port)) != 0:
                return port
    raise RuntimeError(f'No free local port found from {start_port} to {start_port + attempts - 1}')


def tail_log(path='flask_local.log', chars=5000):
    log_path = Path(path)
    if not log_path.exists():
        return '(flask_local.log was not created)'
    content = log_path.read_text(errors='replace')
    return content[-chars:] if content else '(flask_local.log is empty)'

# Stop the previous server started by this notebook, if it exists.
try:
    if flask_server.poll() is None:
        flask_server.terminate()
        try:
            flask_server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            flask_server.kill()
except NameError:
    pass

PORT = find_free_port(PREFERRED_PORT)
log_file = open('flask_local.log', 'w')
flask_server = subprocess.Popen(
    [
        sys.executable,
        '-m',
        'flask',
        '--app',
        'frontend/app.py',
        'run',
        '--host',
        BIND_HOST,
        '--port',
        str(PORT),
        '--no-reload',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    text=True,
    start_new_session=True,
)

Path('flask_local.pid').write_text(str(flask_server.pid))
print('Server process id:', flask_server.pid)
print('Selected port:', PORT)

test_url = f'http://{VISIT_HOST}:{PORT}/classify'
last_error = None
for attempt in range(1, 31):
    if flask_server.poll() is not None:
        print(tail_log())
        raise RuntimeError(f'Flask stopped during startup with return code {flask_server.poll()}')
    try:
        response = urllib.request.urlopen(test_url, timeout=2)
        print('Local test:', response.status, response.reason)
        break
    except Exception as exc:
        last_error = exc
        time.sleep(1)
else:
    print(tail_log())
    raise RuntimeError(f'Flask did not respond after 30 seconds. Last error: {last_error!r}')

# Give the process one extra moment and verify it is still alive after the test request.
time.sleep(1)
if flask_server.poll() is not None:
    print(tail_log())
    raise RuntimeError(f'Flask answered once but then stopped with return code {flask_server.poll()}')

print('\nOpen these links:')
print('Home:', f'http://{VISIT_HOST}:{PORT}/')
print('Classifier:', f'http://{VISIT_HOST}:{PORT}/classify')
print('Generate:', f'http://{VISIT_HOST}:{PORT}/generate')
print('\nAlternative if your browser is outside this Linux environment:')
print('Try replacing 127.0.0.1 with localhost, for example:', f'http://localhost:{PORT}/')
print('\nIf a page shows missing COVID checkpoints, copy covid_dcgan.pth and covid_classifier.pth into checkpoints/.')


Server process id: 2338523
Selected port: 5000
Local test: 200 OK

Open these links:
Home: http://127.0.0.1:5000/
Classifier: http://127.0.0.1:5000/classify
Generate: http://127.0.0.1:5000/generate

Alternative if your browser is outside this Linux environment:
Try replacing 127.0.0.1 with localhost, for example: http://localhost:5000/

If a page shows missing COVID checkpoints, copy covid_dcgan.pth and covid_classifier.pth into checkpoints/.


## Stop Flask

Run this when you are done using the local website.


In [ ]:
import subprocess
import time

try:
    if flask_server.poll() is None:
        flask_server.terminate()
        try:
            flask_server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            flask_server.kill()
        print('Flask stopped.')
    else:
        print('Flask was already stopped. Return code:', flask_server.poll())
except NameError:
    print('No Flask server variable found in this notebook session.')
